# Name That LLM: offline submission (batch inference)

This notebook produces our competition submission. When you submit it, Kaggle **re-runs it
on ~4,000 hidden passages with the internet turned off**. While you edit it, it only sees 3
example test rows. The code must therefore work offline and must not assume a row count.

- **Code:** the project's package, attached as the dataset `ryqn00/name-that-llm-lib` and
  published automatically by CI on each version tag
  ([repo](https://github.com/Ryqn0/name-that-llm)).
- **Model:** the baseline champion, `lr`: word + character TF-IDF with logistic regression.
  It had the best prompt-grouped CV (0.321), and its bootstrap 95% CI against the runner-up
  (NB-SVM) was [+0.008, +0.040], entirely above 0.
- **Post-processing, an A/B test:** with `ASSIGN = True`, the notebook rebuilds the hidden
  prompts from topic words, then lets each model take at most one passage per prompt. On
  CV this raised the score from 0.321 to 0.406. The previous submission (public LB 0.320)
  had `ASSIGN = False`. The model is the same; only this switch changed, so the
  leaderboard difference measures the post-processing alone.

In [ ]:
# Internet is OFF, so install from the attached dataset. Search for the wheel instead of
# hard-coding the mount path: Kaggle's input layout has changed before (the competition data
# now sits under /kaggle/input/competitions/...).
import glob
import os

found = glob.glob("/kaggle/input/**/name_that_llm-*.whl", recursive=True)
wheel_dirs = {os.path.dirname(p) for p in found}
assert len(wheel_dirs) == 1, f"expected exactly one wheel folder, found: {wheel_dirs}"
WHEELS = wheel_dirs.pop()
print("installing from", WHEELS, "->", sorted(os.listdir(WHEELS)))

In [ ]:
# --no-index: never reach for PyPI (there's no internet). --find-links: take wheels from
# the attached folder. Packages Kaggle already has (pandas, scikit-learn, ...) stay as they are.
%pip install -q --no-index --find-links {WHEELS} name-that-llm

In [ ]:
import time

import pandas as pd

import name_that_llm
from name_that_llm.config import load_config
from name_that_llm.data import load_test, load_train
from name_that_llm.submit import predict_submission

CHAMPION = "lr"  # see the baselines notebook: best CV, and the A/B bootstrap confirmed it
ASSIGN = False  # P3a A/B test: True also scored 0.320 on the public LB (no gain), so it stays off
print("package", name_that_llm.__version__, "| champion:", CHAMPION, "| assign:", ASSIGN)

train, test = load_train(), load_test()  # validated: ids stay strings, labels are checked
print("train", train.shape, "| test", test.shape)

In [ ]:
start = time.time()
# Fits on ALL of train (CV was only for measuring), predicts test in order, and VALIDATES
# the result (every id exactly once, no nulls, only the 18 labels) before we write anything.
sub = predict_submission(load_config(CHAMPION), train, test, assign=ASSIGN)
sub.to_parquet("/kaggle/working/submission.parquet", index=False)
print(f"wrote submission.parquet: {len(sub)} rows in {time.time() - start:.0f}s")
sub.head()

## Runtime check

The hidden rerun has ~4,000 passages and a **9-hour** limit. We time a full fit plus a
prediction on 4,000 stand-in rows (training texts, resampled) to be sure we're far below
that limit.

In [ ]:
stand_in = pd.DataFrame(
    {
        "id": [f"t{i}" for i in range(4000)],
        "text": train["text"].sample(4000, replace=True, random_state=0).to_numpy(),
    }
)
start = time.time()
predict_submission(load_config(CHAMPION), train, stand_in, assign=ASSIGN)
minutes = (time.time() - start) / 60
print(f"fit + predict on 4,000 rows: {minutes:.1f} min (limit: 540 min)")
assert minutes < 60, "far too slow for comfort; investigate before submitting"